# Appendix B — Lux and The Cannon with *pollux*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vasilybelokurov/the-cannon-tutorial/blob/main/notebooks/B_pollux.ipynb)

**Goal.** Show how to fit Lux (Chapter 11) and The Cannon (Chapters 2–10) with *pollux*
([github.com/adrn/pollux](https://github.com/adrn/pollux), [documentation](https://pollux-astro.readthedocs.io)),
A. Price-Whelan's JAX framework for data-driven latent-variable models, and check our numpy code against it on the
data of Chapters 10–11.

**You will learn**
1. pollux's data containers, its convention for bad pixels and its label preprocessing;
2. to train a Lux and a Cannon model with pollux and to label new stars from their spectra;
3. that our numpy Lux and pollux's solve the same least-squares problems to $10^{-9}$ of the errors, and that both
   packages give the same label precision for Lux and for The Cannon.

**Versions.** pollux is young and fast-moving (version 0.1.0, August 2026). Its dependencies are not capped, and a
newer release of one of them (`xmmutablemap` 0.3, October 2026) breaks it. This notebook therefore pins every
package it adds (see `requirements.txt` in the repository): pollux 0.1.0, `xmmutablemap` 0.2.2, jax 0.12.0.1,
numpyro 0.22.0, equinox 0.13.8.

In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec('pollux') is None:                  # e.g. in Colab
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pollux==0.1.0', 'xmmutablemap==0.2.2',
                    'jax==0.12.0.1', 'jaxlib==0.12.0.1', 'numpyro==0.22.0', 'equinox==0.13.8'], check=True)

In [ ]:
import hashlib, json, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- Data access (identical in every chapter) -------------------------------------------------------------
# The data files are extracts of APOGEE DR17 and Gaia DR3, published on Zenodo (https://doi.org/10.5281/zenodo.23813051).
# MANIFEST.json lists, for each file, the query that made it and its SHA-256 checksum.
DATA_URL = 'https://zenodo.org/records/23813051/files/{}?download=1'
_local = Path('../data/MANIFEST.json')            # present when you run inside a clone of the repository
MANIFEST = json.loads(_local.read_text() if _local.exists() else urllib.request.urlopen(DATA_URL.format('MANIFEST.json')).read())


def load(name, cache=Path('data')):
    """Download <name> (.parquet table or .npz arrays) from Zenodo if needed, check its SHA-256, read it.

    A .parquet file is returned as a pandas DataFrame, an .npz file as a dict of numpy arrays.
    """
    ext = '.npz' if name.startswith(('spectra_', 'rvs_')) else '.parquet'
    path = cache / f'{name}{ext}'
    sha = lambda p: hashlib.sha256(p.read_bytes()).hexdigest()
    if not path.exists() or sha(path) != MANIFEST[name]['sha256']:   # first use, or a stale/damaged copy
        cache.mkdir(exist_ok=True)
        print(f'downloading {path.name} ...')
        part = path.with_suffix('.part')
        urllib.request.urlretrieve(DATA_URL.format(path.name), part)   # download to a temporary file first
        if sha(part) != MANIFEST[name]['sha256']:
            raise IOError(f'{path.name}: checksum does not match the manifest (incomplete download?)')
        part.replace(path)                                         # only a verified file takes the final name
    # the checksum guarantees that everyone runs this chapter on exactly the same data
    if ext == '.parquet':
        return pd.read_parquet(path)
    with np.load(path, allow_pickle=False) as f:   # allow_pickle=False: arrays only, no code can be loaded
        return {k: f[k] for k in f.files}
# --- Spectra and continuum normalisation (as in Chapters 0 and 3) ----------------------------------------
NPIX = 8575
wave = 10**(4.179 + 6e-6 * np.arange(NPIX))               # apStar vacuum wavelength grid [Angstrom]
CHIPS = [(15150, 15800), (15865, 16425), (16480, 16950)]  # the three detectors [A], inside the edges of Chapter 0
BAD_BITS = sum(1 << b for b in [0, 1, 2, 3, 4, 5, 6, 7, 12])   # pixels given zero weight (Chapter 0)


def prepare(sp, rows=slice(None)):
    """flux and err (float64) of the spectra file sp; err = inf where there are no data, a BAD_BITS flag, or
    outside CHIPS (the detector edges, where only some spectra have data)."""
    flux = sp['flux'][rows].astype(float)
    err = sp['err'][rows].astype(float)
    on_chip = np.any([(wave > lo) & (wave < hi) for lo, hi in CHIPS], axis=0)   # pixels at the detector edges and
    bad = ((sp['mask'][rows] & BAD_BITS) > 0) | ~np.isfinite(flux) | ~np.isfinite(err) | (err <= 0) | ~on_chip  # gaps: no weight
    return np.where(bad, 0.0, flux), np.where(bad, np.inf, err)


def cheb_normalise(flux, err, cont, order=2):
    """Divide by a Chebyshev polynomial of the given order fitted, per detector, to the continuum pixels `cont`
    (boolean, NPIX) with weights 1/err^2 (N15 sect. 2.3 and 5.3). Returns normalised flux and error;
    pixels outside the detectors get err = inf."""
    fn, en = np.zeros_like(flux), np.full_like(err, np.inf)
    for lo, hi in CHIPS:
        c = (wave > lo) & (wave < hi)
        t = (wave[c] - (lo + hi) / 2) / ((hi - lo) / 2)          # wavelength mapped to [-1, 1]
        B = np.polynomial.chebyshev.chebvander(t, order)         # (pixels, order+1) basis
        use = cont[c]
        w = 1 / err[:, c][:, use]**2                             # (N, continuum pixels); 0 for flagged pixels
        # one small weighted least-squares problem per star: (B^T W B) a = B^T W f
        BtWB = np.einsum('pi,np,pj->nij', B[use], w, B[use])
        BtWf = np.einsum('pi,np->ni', B[use], w * flux[:, c][:, use])
        a = np.linalg.solve(BtWB, BtWf[:, :, None])[:, :, 0]     # (N, order+1); b as a stack of column vectors
        cont_fit = a @ B.T                                       # (N, pixels of this detector)
        fn[:, c] = flux[:, c] / cont_fit
        en[:, c] = err[:, c] / cont_fit
    return fn, en
# --- The Cannon (as written in Chapter 2) -----------------------------------------------------------------
from scipy.optimize import least_squares


def vectorize(x):
    """Quadratic label vector v(x) = [1, x_k, x_k x_l (k <= l)] for scaled labels x of shape (N, K): (N, D)."""
    x = np.atleast_2d(x)
    k, l = np.triu_indices(x.shape[1])
    return np.hstack([np.ones((len(x), 1)), x, x[:, k] * x[:, l]])


def dvectorize(x):
    """Derivative dv/dx of the label vector of one star (x of shape (K,)): (D, K)."""
    K = len(x)
    k, l = np.triu_indices(K)
    J = np.zeros((1 + K + len(k), K))
    J[1:1 + K] = np.eye(K)
    for i, (a, b) in enumerate(zip(k, l)):
        J[1 + K + i, a] += x[b]
        J[1 + K + i, b] += x[a]
    return J


def fit_theta(V, flux, var):
    """Weighted least squares at every pixel: theta (P, D) and its covariance (P, D, D); weights 1/var (N, P).
    Zero-weight entries (var = inf) drop out of the sums."""
    w = 1 / var
    flux = np.where(w > 0, flux, 0.0)                       # NaN flux at zero weight must not poison the sums
    N, Dv = V.shape
    A = ((V[:, :, None] * V[:, None, :]).reshape(N, Dv * Dv).T @ w).T.reshape(-1, Dv, Dv)
    b = (V.T @ (w * flux)).T
    cov = np.linalg.pinv(A)                                 # pinv: pixels without data give theta = 0, not an error
    return np.einsum('pde,pe->pd', cov, b), cov


S_GRID = np.r_[0, np.logspace(-4, 0, 41)]                  # 0, then 1e-4 ... 1 in steps of 0.1 dex


def fit_scatter(resid, err2):
    """Maximum-likelihood scatter s per pixel for fixed residuals: best value on the grid S_GRID, refined by a
    parabola in log s (Chapter 2). Zero-weight entries (err2 = inf) are ignored."""
    use = np.isfinite(err2) & np.isfinite(resid)
    r2 = np.where(use, resid**2, 0.0)
    e2 = np.where(use, err2, 1.0)
    nll = np.array([np.sum(use * (r2 / (e2 + s**2) + np.log(e2 + s**2)), axis=0) for s in S_GRID])
    best = np.argmin(nll, axis=0)
    j = np.clip(best, 2, len(S_GRID) - 2)
    p = np.arange(nll.shape[1])
    a, b, c = nll[j - 1, p], nll[j, p], nll[j + 1, p]
    shift = np.clip(0.5 * (a - c) / np.where(a - 2 * b + c > 0, a - 2 * b + c, np.inf), -1, 1)   # stay near the grid optimum
    return np.where(best == 0, 0.0, S_GRID[j] * np.exp(shift * np.log(S_GRID[2] / S_GRID[1])))


def train(x, flux, err, n_iter=3):
    """Training step. x: scaled labels (N, K); flux, err: (N, P) (err = inf for pixels to ignore).
    Returns theta (P, D), its covariance (P, D, D) and the scatter s (P,)."""
    V, err2 = vectorize(x), err**2
    s = np.zeros(flux.shape[1])
    for _ in range(n_iter):
        theta, cov = fit_theta(V, flux, err2 + s**2)
        s = fit_scatter(flux - V @ theta.T, err2)
    theta, cov = fit_theta(V, flux, err2 + s**2)
    return theta, cov, s


def test(flux, err, theta, s, x0=None):
    """Test step for one spectrum: scaled labels (K,), covariance (K, K), chi^2, number of pixels used, converged?"""
    K = int(round((np.sqrt(8 * theta.shape[1] + 1) - 3) / 2))   # invert D = 1 + K + K(K+1)/2
    ok = np.isfinite(flux) & np.isfinite(err) & (theta[:, 0] != 0)
    w = 1 / np.sqrt(err[ok]**2 + s[ok]**2)
    th, f = theta[ok], flux[ok]
    res = lambda x: (f - th @ vectorize(x)[0]) * w
    jac = lambda x: -(th @ dvectorize(x)) * w[:, None]
    r = least_squares(res, np.zeros(K) if x0 is None else x0, jac=jac, method='lm')
    # covariance from the Jacobian at the optimum (pinv: no crash if a label is unconstrained); r.success: converged
    return r.x, np.linalg.pinv(r.jac.T @ r.jac), np.sum(r.fun**2), ok.sum(), r.success


def train_clean(x, flux, err, cut=3.0):
    """Train, drop the training stars whose mean squared residual per pixel (in units of sigma^2 + s^2) exceeds
    `cut`, and train again (Chapter 4). Returns theta, its covariance, s, and the boolean mask of stars kept."""
    theta, cov, s = train(x, flux, err)
    z2 = (flux - vectorize(x) @ theta.T)**2 / (err**2 + s**2)
    z2 = np.where(np.isfinite(err) & (theta[:, 0] != 0), z2, np.nan)
    keep = np.nanmean(z2, axis=1) < cut
    theta, cov, s = train(x[keep], flux[keep], err[keep])
    return theta, cov, s, keep


def scatter_stats(d):
    """Summary of differences d (e.g. Cannon - reference): mean (bias), standard deviation, root-mean-square error,
    robust standard deviation 1.4826 x median absolute deviation (= std for a Gaussian, insensitive to outliers),
    and the fraction of points more than 3 robust std from the median (the tails)."""
    d = np.asarray(d)[np.isfinite(d)]
    med = np.median(d)
    rob = 1.4826 * np.median(np.abs(d - med))
    return {'bias': d.mean(), 'std': d.std(), 'rmse': np.sqrt(np.mean(d**2)), 'robust': rob,
            'tail': np.mean(np.abs(d - med) > 3 * rob)}
# --- More of The Cannon (Chapter 10): any polynomial order, censoring, L1 regularisation, several starting points,
# --- and a test of whether labels lie inside the training set -------------------------------------------------
from itertools import combinations_with_replacement
import warnings
from scipy.optimize import linprog


def label_terms(K, order=2):
    """The terms of the label vector after the leading 1: tuples of label indices, e.g. (0,), (0, 1), (2, 2, 2)."""
    return [t for p in range(1, order + 1) for t in combinations_with_replacement(range(K), p)]


def vectorize_n(x, order=2):
    """Label vector of polynomial order `order` for scaled labels x (N, K): (N, D). For order 2 it equals vectorize."""
    x = np.atleast_2d(x)
    return np.hstack([np.ones((len(x), 1))] + [np.prod(x[:, list(t)], axis=1, keepdims=True)
                                                for t in label_terms(x.shape[1], order)])


def dvectorize_n(x, order=2):
    """Derivative dv/dx of the label vector of one star (x of shape (K,)): (D, K)."""
    T = label_terms(len(x), order)
    J = np.zeros((1 + len(T), len(x)))
    for i, t in enumerate(T):
        for j in range(len(t)):                              # d/dx of x_a x_b x_c ...: drop one factor at a time
            J[1 + i, t[j]] += np.prod(x[list(t[:j] + t[j + 1:])])
    return J


def censor_mask(allowed, order=2):
    """Coefficients allowed at each pixel, (P, D), from the labels allowed at each pixel, (P, K): a term may be
    non-zero only if all the labels in it are allowed. The leading coefficient (the mean spectrum) always is."""
    T = label_terms(allowed.shape[1], order)
    return np.hstack([np.ones((len(allowed), 1), bool)] + [allowed[:, list(t)].all(axis=1, keepdims=True) for t in T])


def fit_theta_l1(V, flux, var, lam=0.0, mask=None, theta0=None, chunk=2048, tol=1e-8, max_iter=20000, kappa=0.1):
    """At every pixel minimise chi^2 + lam * sum_{d >= 1} |theta_d| (Casey et al. 2016), with theta_d = 0 where
    mask (P, D) is False. lam = 0: weighted least squares, solved directly. lam > 0: ADMM (Boyd et al. 2011, §6.4) on rescaled coefficients,
    all pixels of a chunk at once, started from theta0 (P, D) if given. Weights 1/var (N, P); zero-weight entries drop
    out. Returns theta (P, D) and the number of ADMM iterations used (0 for lam = 0)."""
    N, D = V.shape
    P = flux.shape[1]
    mask = np.ones((P, D), bool) if mask is None else mask
    w = 1 / var
    flux = np.where(w > 0, flux, 0.0)
    VV = (V[:, :, None] * V[:, None, :]).reshape(N, D * D).T
    pen = np.r_[0.0, np.ones(D - 1)]                             # theta_0 (the mean spectrum) is not penalised
    theta, n_used = np.zeros((P, D)), 0
    for p0 in range(0, P, chunk):
        sl = slice(p0, p0 + chunk)
        A = (VV @ w[:, sl]).T.reshape(-1, D, D)                  # V^T W V at each pixel
        b = (V.T @ (w[:, sl] * flux[:, sl])).T                   # V^T W f
        m = mask[sl]
        if lam == 0:                                             # censored rows/columns replaced by the identity
            Am = A * (m[:, :, None] & m[:, None, :]) + np.eye(D) * ~m[:, None, :]
            theta[sl] = np.einsum('pde,pe->pd', np.linalg.pinv(Am), b * m) * m
            continue
        # Rescale each coefficient so that the curvature of chi^2 along it is 1: theta = d * y. Without this, linear and
        # quadratic terms have curvatures that differ by orders of magnitude and ADMM crawls. |theta_k| = d_k |y_k|.
        dg = np.einsum('pdd->pd', A)
        d = 1 / np.sqrt(np.where(dg > 0, 2 * dg, 1.0))
        As, bs = A * d[:, :, None] * d[:, None, :], b * d
        # ADMM: split y (quadratic part) from z (L1 part and mask), y = z; u is the scaled dual variable. The step
        # size rho scales with the soft threshold lam * d, which in these units sets how far the solution moves.
        rho = np.minimum(kappa * lam * np.median(d, axis=1), 1.0)[:, None]   # capped at the (unit) curvature
        M = np.linalg.inv(2 * As + rho[:, :, None] * np.eye(D))   # factorised once, used at every iteration
        z = np.zeros_like(b) if theta0 is None else theta0[sl] * m / d
        u = np.zeros_like(b)
        thr = pen * lam * d / rho                                # soft threshold for each scaled coefficient
        act = np.arange(len(b))                                  # pixels not yet converged
        for it in range(max_iter):
            y = np.einsum('pde,pe->pd', M[act], 2 * bs[act] + rho[act] * (z[act] - u[act]))
            v = y + u[act]
            z_new = np.where(m[act], np.sign(v) * np.maximum(np.abs(v) - thr[act], 0), 0.0)
            da = d[act]                                          # convergence judged on theta, not on y
            done = (np.abs(da * (y - z_new)).max(axis=1) < tol) & (np.abs(da * (z_new - z[act])).max(axis=1) < tol)
            z[act], u[act] = z_new, v - z_new
            act = act[~done]
            if len(act) == 0:
                break
        if len(act):
            warnings.warn(f'fit_theta_l1: {len(act)} pixels did not converge in {max_iter} iterations')
        n_used = max(n_used, it + 1)
        theta[sl] = z * d                                        # z has the exact zeros
    return theta, n_used


def median_scatter(resid, err2, n_bisect=50):
    """AnniesLasso's scatter: at each pixel the s >= 0 for which the median of resid^2 / (err2 + s^2) over the stars is
    1 (s = 0 where it is already below 1). Bisection on all pixels at once; zero-weight entries are ignored."""
    z = lambda s: np.nanmedian(np.where(np.isfinite(err2), resid**2 / (err2 + s**2), np.nan), axis=0)
    lo, hi = np.zeros(resid.shape[1]), np.full(resid.shape[1], 1.0)
    for _ in range(n_bisect):                                # median chi^2 decreases with s
        mid = (lo + hi) / 2
        big = z(mid) > 1
        lo, hi = np.where(big, mid, lo), np.where(big, hi, mid)
    return np.where(z(0.0) > 1, hi, 0.0)


def train_more(x, flux, err, lam=0.0, mask=None, order=2, n_iter=3, scatter='ml'):
    """Training step with polynomial order, censoring mask and L1 regularisation. scatter='ml': coefficients and
    maximum-likelihood scatter iterated, as in `train` (Chapter 2). scatter='median': AnniesLasso's recipe —
    coefficients fitted once with s = 0, then s from `median_scatter`. Returns theta (P, D) and s (P,)."""
    V, err2 = vectorize_n(x, order), err**2
    if scatter == 'median':
        theta = fit_theta_l1(V, flux, err2, lam, mask)[0]
        return theta, median_scatter(flux - V @ theta.T, err2)
    s, theta = np.zeros(flux.shape[1]), None
    for _ in range(n_iter):
        theta = fit_theta_l1(V, flux, err2 + s**2, lam, mask, theta)[0]     # each fit starts from the previous one
        s = fit_scatter(flux - V @ theta.T, err2)
    return fit_theta_l1(V, flux, err2 + s**2, lam, mask, theta)[0], s


def test_more(flux, err, theta, s, order=2, starts=None):
    """Test step for one spectrum, as `test` (Chapter 2) but for any polynomial order and from several starting
    points (rows of `starts`, scaled labels; default: the median labels, 0); of the fits that converged, the one with
    the lowest chi^2 is kept.
    Returns scaled labels, covariance, chi^2, number of pixels used, converged?"""
    K = next(k for k in range(1, 100) if 1 + len(label_terms(k, order)) == theta.shape[1])
    ok = np.isfinite(flux) & np.isfinite(err) & (theta[:, 0] != 0)
    w = 1 / np.sqrt(err[ok]**2 + s[ok]**2)
    th, f = theta[ok], flux[ok]
    res = lambda x: (f - th @ vectorize_n(x, order)[0]) * w
    jac = lambda x: -(th @ dvectorize_n(x, order)) * w[:, None]
    fits = [least_squares(res, x0, jac=jac, method='lm') for x0 in (np.zeros((1, K)) if starts is None else
                                                                     np.atleast_2d(starts))]
    r = min(fits, key=lambda r: (not r.success, np.sum(r.fun**2)))   # converged fits first, then lowest chi^2
    return r.x, np.linalg.pinv(r.jac.T @ r.jac), np.sum(r.fun**2), ok.sum(), r.success


def in_hull(x, X):
    """Is each row of x (M, K) inside the convex hull of the training labels X (N, K)? True if x is a weighted mean
    of training points with non-negative weights summing to 1: a linear-programming feasibility problem, which works
    in any number of dimensions (a Delaunay triangulation does not, beyond a few). Labels are scaled by the standard
    deviation of X so that kelvin and dex are treated alike."""
    sd = X.std(axis=0)
    A = np.vstack([(X / sd).T, np.ones(len(X))])
    status = np.array([linprog(np.zeros(len(X)), A_eq=A, b_eq=np.r_[p / sd, 1], bounds=(0, None), method='highs').status
                       for p in np.atleast_2d(x)])
    if not np.all((status == 0) | (status == 2)):              # 0: feasible (inside), 2: infeasible (outside)
        raise RuntimeError(f'linprog failed with status {set(status) - {0, 2}}')
    return status == 0
# --- Lux (Horta et al. 2025): labels and spectra both generated from a latent vector per star (Chapter 11) ----
# labels l_n = A z_n + noise (errors sl_n), fluxes f_n = B z_n + noise (errors s_n and pixel scatter s); every step of
# the training is a weighted linear least-squares problem, solved here in closed form.


def lux_weights(values, errors):
    """Inverse variances, 0 where the error is not finite and positive or the value is not finite (masked pixels,
    missing labels), and the values with those entries set to 0 so that they cannot produce NaN."""
    ok = np.isfinite(errors) & (errors > 0) & np.isfinite(values)
    w = np.where(ok, 1 / np.where(ok, errors, 1.0)**2, 0.0)
    return w, np.where(w > 0, values, 0.0)


def lsq_update(G, h, c0, incremental=True, rcond=1e-10):
    """Solve the normal equations G c = h for a stack of problems (G: (K, P, P), h and c0: (K, P)).
    incremental=True: c0 + G^+ (h - G c0), which has the same residual as the minimum-norm solution but keeps the part
    of c0 that the data do not constrain (as a Gauss-Newton step from c0 does); incremental=False: G^+ h.
    Directions with eigenvalues of G below rcond times the largest count as unconstrained (rcond = 0: none do, for G
    that is positive definite, e.g. with a prior)."""
    vals, vecs = np.linalg.eigh(G)
    inv = np.where(vals > rcond * np.max(vals, axis=1, keepdims=True), 1 / np.where(vals > 0, vals, 1.0), 0.0)
    r = h - np.einsum('kpq,kq->kp', G, c0) if incremental else h
    step = np.einsum('kpq,kq,kq->kp', vecs, inv, np.einsum('kqp,kq->kp', vecs, r))
    return c0 + step if incremental else step


def lux_a_step(Z, lab, wl, A, incremental=True):
    """Label coefficients A (M, P) at fixed latents Z (N, P): one weighted least-squares problem per label."""
    G = np.einsum('nm,np,nq->mpq', wl, Z, Z)
    return lsq_update(G, np.einsum('nm,np->mp', wl * lab, Z), A, incremental)


def lux_b_step(Z, flux, w, B, incremental=True):
    """Flux coefficients B (Lambda, P) at fixed latents Z: one weighted least-squares problem per pixel."""
    N, P = Z.shape
    G = ((Z[:, :, None] * Z[:, None, :]).reshape(N, P * P).T @ w).T.reshape(-1, P, P)
    return lsq_update(G, (Z.T @ (w * flux)).T, B, incremental)


def lux_z_step(A, B, lab, wl, flux, w, Z, tau=np.inf, incremental=True):
    """Latents Z (N, P) at fixed A, B: one weighted least-squares problem per star, using its labels (weights wl) and
    fluxes (weights w), with a Gaussian prior of width tau on every latent (ridge term)."""
    P = A.shape[1]
    G = (w @ (B[:, :, None] * B[:, None, :]).reshape(-1, P * P)).reshape(-1, P, P)
    G += np.einsum('nm,mp,mq->npq', wl, A, A) + np.eye(P) / tau**2
    rcond = 0.0 if np.isfinite(tau) else 1e-10                # with a prior every direction is constrained by it
    return lsq_update(G, (w * flux) @ B + (wl * lab) @ A, Z, incremental, rcond)


def lux_objective(A, B, Z, s, lab, wl, flux, w, tau=np.inf):
    """Q = -2 ln L (labels and fluxes, up to a constant) + sum |z|^2 / tau^2; masked entries (weight 0) excluded."""
    var = np.where(w > 0, 1 / np.where(w > 0, w, 1.0) + s**2, 1.0)
    q_flux = np.where(w > 0, (flux - Z @ B.T)**2 / var + np.log(var), 0.0)
    return np.sum(wl * (lab - Z @ A.T)**2) + np.sum(q_flux) + np.sum(Z**2) / tau**2


def lux_scatter_step(B, Z, flux, w, s):
    """Pixel scatter by maximum likelihood at fixed B, Z (`fit_scatter` of Chapter 2), kept at each pixel only if it
    does not increase that pixel's part of Q."""
    err2 = np.where(w > 0, 1 / np.where(w > 0, w, 1.0), np.inf)
    resid = flux - Z @ B.T
    q = lambda s_: np.sum(np.where(w > 0, resid**2 / (err2 + s_**2) + np.log(np.where(w > 0, err2, 1.0) + s_**2), 0.0),
                          axis=0)
    s_new = fit_scatter(resid, err2)
    return np.where(q(s_new) <= q(s), s_new, s)


def lux_init(lab, P, n_flux, seed=0):
    """H25 Table 1, eq. 9: z_n = [1, (l_n - median) / ((p97.5 - p2.5) / 4), 0, ...], A and B uniform in [0, 1].
    Pivot and scale come from the labels that are present (NaN = missing); missing entries start at 0."""
    rng = np.random.default_rng(seed)
    q = np.nanpercentile(lab, [2.5, 50, 97.5], axis=0)
    x = np.nan_to_num((lab - q[1]) / ((q[2] - q[0]) / 4))
    Z = np.hstack([np.ones((len(lab), 1)), x, np.zeros((len(lab), P - lab.shape[1] - 1))])
    return rng.uniform(0, 1, (lab.shape[1], P)), rng.uniform(0, 1, (n_flux, P)), Z


def lux_train(lab, lab_err, flux, err, P=49, n_iter=5, tau=1000.0, incremental=True, seed=0):
    """Training as described in H25 §4.2. Stage 1: n_iter rounds of A-, B- and z-steps without scatter or prior.
    Stage 2 (once): scatter, then B with the new scatter, then z with the new B, scatter and the prior.
    Labels missing as NaN or with infinite error are ignored. Returns A, B, s, Z and Q after every step."""
    wl, l0 = lux_weights(lab, lab_err)
    w, f0 = lux_weights(flux, err)
    A, B, Z = lux_init(np.where(wl > 0, lab, np.nan), P, flux.shape[1], seed)
    s, history = np.zeros(flux.shape[1]), []
    for _ in range(n_iter):
        A = lux_a_step(Z, l0, wl, A, incremental)
        B = lux_b_step(Z, f0, w, B, incremental)
        Z = lux_z_step(A, B, l0, wl, f0, w, Z, np.inf, incremental)
        history.append(lux_objective(A, B, Z, s, l0, wl, f0, w))
    s = lux_scatter_step(B, Z, f0, w, s)
    wv = w / (1 + w * s**2)                                   # 1 / (sigma^2 + s^2), 0 where masked
    B = lux_b_step(Z, f0, wv, B, incremental)
    Z = lux_z_step(A, B, l0, wl, f0, wv, Z, tau, incremental)
    history.append(lux_objective(A, B, Z, s, l0, wl, f0, w, tau))
    return A, B, s, Z, np.array(history)


def lux_test(flux, err, A, B, s, rcond=1e-12):
    """Test step (H25 eq. 10): latents of each star from its spectrum alone (no prior), labels = A z.
    Latent directions the spectrum does not constrain (eigenvalues of B^T W B below rcond times the largest) are left at
    zero; the labels do not depend on them only if A has no component along them, which is checked and returned.
    Returns labels (N, M), their formal covariance (N, M, M), and for each star the condition number of the retained
    part of B^T W B and the largest |A v| over the discarded directions v (0 if none)."""
    w, f0 = lux_weights(flux, err)
    wv = w / (1 + w * s**2)
    P = B.shape[1]
    G = (wv @ (B[:, :, None] * B[:, None, :]).reshape(-1, P * P)).reshape(-1, P, P)
    vals, vecs = np.linalg.eigh(G)
    keep = vals > rcond * vals[:, -1:]
    Ginv = np.einsum('npq,nq,nrq->npr', vecs, np.where(keep, 1 / np.where(keep, vals, 1.0), 0.0), vecs)
    z = np.einsum('npq,nq->np', Ginv, (wv * f0) @ B)
    cond = vals[:, -1] / np.min(np.where(keep, vals, np.inf), axis=1)
    leak = np.max(np.where(~keep[:, None, :], np.abs(np.einsum('mp,npq->nmq', A, vecs)), 0.0), axis=(1, 2))
    return z @ A.T, np.einsum('mp,npq,lq->nml', A, Ginv, A), cond, leak
import time
import warnings
import jax
import numpyro
import numpyro.distributions as dist
import pollux as plx
from pollux.models.transforms import LinearTransform

jax.config.update('jax_enable_x64', True)                       # double precision (JAX defaults to single)
warnings.filterwarnings('ignore', module='pollux')
labels = load('labels')
cont = load('continuum_pixels')['continuum'].to_numpy()
NAMES = ['teff', 'logg', 'fe_h'] + [e + '_fe' for e in ['c', 'n', 'o', 'mg', 'al', 'si', 'ca', 'mn', 'ni']]
data = {}
for nm in ['train', 'test']:
    lab = labels[labels['sample'] == nm].reset_index(drop=True)
    f, e = cheb_normalise(*prepare(load(f'spectra_{nm}')), cont)
    good = np.isfinite(lab[NAMES]).all(axis=1).to_numpy()
    data[nm] = (lab[NAMES].to_numpy(float)[good], lab[[n + '_err' for n in NAMES]].to_numpy(float)[good], f[good], e[good])
L, Lerr, flux, err = data['train']
Lt, _, ft, et = (a[::2] for a in data['test'])
val = np.random.default_rng(10).permutation(len(L)) % 5 == 0  # the splits of Chapters 10 and 11
tr = np.flatnonzero(~val)
rob = lambda d: np.array([scatter_stats(d[:, k])['robust'] for k in range(d.shape[1])])
print(f'pollux {plx.__version__ if hasattr(plx, "__version__") else ""}, jax {jax.__version__}, numpyro {numpyro.__version__}')

## B.1 Data in pollux

pollux keeps each observed quantity in an `OutputData` (values and errors) and the outputs of a set of stars in a
`PolluxData`, keyed by name — for Lux, `"flux"` and `"label"`. Bad pixels are marked by a flux of 1 and a huge error
(we use $\infty$ elsewhere). Labels can be scaled by a preprocessor, which matters for the default $\mathcal N(0, 1)$
priors and is undone when predicting.

In [ ]:
def pdata(F, E, labs=None, labs_err=None, pre=None):
    """PolluxData with fluxes (bad pixels: flux 1, error 1e10) and optionally labels (scaled by pre)."""
    bad = ~np.isfinite(E)
    out = {'flux': plx.data.OutputData(np.where(bad, 1.0, F), err=np.where(bad, 1e10, E))}
    if labs is not None:
        out['label'] = plx.data.OutputData(labs, err=labs_err, preprocessor=pre)
    return plx.data.PolluxData(**out)


pre = plx.data.ShiftScalePreprocessor.from_data(L[tr])
train = pdata(flux[tr], err[tr], L[tr], Lerr[tr], pre).preprocess()
print(train)

## B.2 The same least-squares steps

Before comparing models trained each in its own way, check that the building blocks agree. Start our code and
pollux from the same random latents (49 per star) and coefficients, make pollux's priors negligible (very wide priors
on the coefficients, none on the latents), and run five rounds of the same three steps: label coefficients, flux
coefficients, latents. With random latents of full rank every step has a unique solution, so the two must agree.
This checks the least-squares fits of training stage 1 only — not the zero-padded start of Chapter 11, the scatter,
the prior or the test step. (pollux marks bad pixels with a tiny weight, we with zero.)

In [ ]:
P = 49
rng = np.random.default_rng(1)
Z0, A0, B0 = rng.normal(0, 1, (len(tr), P)), rng.uniform(0, 1, (len(NAMES), P)), rng.uniform(0, 1, (flux.shape[1], P))
wl, l0 = lux_weights(L[tr], Lerr[tr])
w, f0 = lux_weights(flux[tr], err[tr])
t0 = time.time()
A, B, Z = A0, B0, Z0
for _ in range(5):
    A = lux_a_step(Z, l0, wl, A)
    B = lux_b_step(Z, f0, w, B)
    Z = lux_z_step(A, B, l0, wl, f0, w, Z)
t_ours = time.time() - t0
m = plx.LVM(latent_size=P)
m.register_output('label', LinearTransform(output_size=len(NAMES), priors={'A': dist.Normal(0, 1e8)}))
m.register_output('flux', LinearTransform(output_size=flux.shape[1], priors={'A': dist.Normal(0, 1e8)}))
t0 = time.time()
res = m.optimize_iterative(pdata(flux[tr], err[tr], L[tr], Lerr[tr]), blocks=['label:data', 'flux:data', 'latents'],
                           initial_params={'latents': Z0, 'label': {'data': {'A': A0}}, 'flux': {'data': {'A': B0}}},
                           max_cycles=5, tol=0.0, latents_prior=False, progress=False)
t_pollux = time.time() - t0
pA, pB, pZ = (np.asarray(x) for x in (res.params['label']['data']['A'], res.params['flux']['data']['A'],
                                      res.params['latents']))
use = np.isfinite(err[tr])
d_lab = np.abs(Z @ A.T - pZ @ pA.T) / Lerr[tr]
d_flux = np.abs(Z @ B.T - pZ @ pB.T)[use] / err[tr][use]
print(f'5 rounds: ours {t_ours:.0f} s, pollux {t_pollux:.0f} s; |difference| / error: labels max {d_lab.max():.1e}, '
      f'fluxes max {d_flux.max():.1e}')
assert d_lab.max() < 1e-6 and d_flux.max() < 1e-6

## B.3 Lux in pollux

`plx.Lux` builds the two linear outputs and a fitted scatter for the flux. `optimize_iterative` alternates its blocks
until the loss stops improving; each linear block is solved exactly, as in our code. pollux's defaults differ from
H25's recipe (Chapter 11): random starting latents, $\mathcal N(0, 1)$ priors on latents and coefficients, and as many
rounds as needed. To label new stars, hold everything but the latents fixed and solve for the latents from the
spectra alone — with pollux's default $\mathcal N(0, 1)$ prior on them (the most probable latents), where our test
step uses no prior.

In [ ]:
t0 = time.time()
lux_px = plx.Lux(latent_size=P, label_size=len(NAMES), flux_size=flux.shape[1])
res = lux_px.optimize_iterative(train, rng_key=jax.random.PRNGKey(0), progress=False)
fixed = lux_px.output_pars(res.params)


def lux_px_labels(F, E):
    """Latents from the spectra alone, then labels, unscaled."""
    r = lux_px.optimize_iterative(pdata(F, E), fixed_pars=fixed, blocks=['latents'], progress=False)
    return np.asarray(pre.inverse_transform(lux_px.predict_outputs(fixed, latents=r.params['latents'])['label']))


px_lux = rob(lux_px_labels(flux[val], err[val]) - L[val]), rob(lux_px_labels(ft, et) - Lt)
print(f'pollux Lux: {len(res.losses_per_cycle)} rounds, {time.time() - t0:.0f} s')
t0 = time.time()
A, B, s, Z, _ = lux_train(L[tr], Lerr[tr], flux[tr], err[tr], P=P)
our_lux = rob(lux_test(flux[val], err[val], A, B, s)[0] - L[val]), rob(lux_test(ft, et, A, B, s)[0] - Lt)
print(f'our Lux: {time.time() - t0:.0f} s')

## B.4 The Cannon in pollux

`plx.Cannon` is a polynomial in the labels, which are its latents: during training they are held fixed at the
(scaled) catalogue values, the coefficients are solved exactly and the scatter by gradient descent (Adam), as in
pollux's APOGEE example. The test step fits the (scaled) labels of each star by gradient descent, with pollux's
default $\mathcal N(0, 1)$ prior on them.

In [ ]:
t0 = time.time()
can_px = plx.Cannon(label_size=len(NAMES), output_size=flux.shape[1], poly_degree=2, intrinsic_scatter={'flux': 0.1})
res = can_px.optimize_iterative(train, blocks=['flux:data', 'flux:err'], fixed_pars={'latents': np.asarray(train['label'].data)},
                                max_cycles=8, rng_key=jax.random.PRNGKey(123), progress=False,
                                block_options={'flux:err': {'optimizer': numpyro.optim.Adam,
                                                            'optimizer_kwargs': {'step_size': 1e-2}, 'num_steps': 1000}})
fixed_c = can_px.output_pars(res.params)


def cannon_px_labels(F, E):
    """Labels of each star by gradient descent (flux likelihood and the default prior on the scaled labels)."""
    pars, _ = can_px.optimize(pdata(F, E), fixed_pars=fixed_c, names=['flux'], num_steps=4000,
                              optimizer=numpyro.optim.Adam(1e-2), rng_key=jax.random.PRNGKey(42), progress=False)
    return np.asarray(pre.inverse_transform(np.asarray(pars['latents'])))


px_can = rob(cannon_px_labels(flux[val], err[val]) - L[val]), rob(cannon_px_labels(ft, et) - Lt)
print(f'pollux Cannon: {time.time() - t0:.0f} s')
t0 = time.time()
pv, sc = np.median(L[tr], axis=0), (np.percentile(L[tr], 97.5, axis=0) - np.percentile(L[tr], 2.5, axis=0)) / 2
th, s = train_more((L[tr] - pv) / sc, flux[tr], err[tr])
run = lambda F, E: np.array([test_more(F[n], E[n], th, s)[0] for n in range(len(F))]) * sc + pv
our_can = rob(run(flux[val], err[val]) - L[val]), rob(run(ft, et) - Lt)
print(f'our Cannon: {time.time() - t0:.0f} s')

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4.2))
models = {'Cannon, ours': our_can, 'Cannon, pollux': px_can, 'Lux, ours': our_lux, 'Lux, pollux': px_lux}
for a, (k, lab) in zip(ax, [(0, r'$T_{\rm eff}$ [K]'), (1, r'$\log g$'), (None, 'median of the nine [X/Fe]')]):
    for j, split in enumerate(['validation (high S/N)', 'test (S/N 50–100)']):
        v = [r[j][k] if k is not None else np.median(r[j][3:]) for r in models.values()]
        a.bar(np.arange(4) + 0.38 * j - 0.19, v, 0.36, color=['C0', 'C1'][j], label=split)
    a.set_xticks(range(4), [m.replace(', ', '\n') for m in models], fontsize=9)
    a.set(ylabel=f'robust std vs ASPCAP, {lab}')
ax[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

**Figure B.1.** The same comparison of $T_{\rm eff}$, $\log g$ and the abundances as Figure 11.1, for our numpy code
and for pollux, each trained its own way on the same 1308 stars. The two Cannons agree to about a kelvin and a few
thousandths of a dex; the two Luxes closely, pollux's being somewhat better in $T_{\rm eff}$ at low S/N — its priors,
random start and training schedule all differ from ours, and we have not separated their effects. The precisions on
which Chapter 11's comparison of Lux and The Cannon rests are reproduced by pollux to within the tolerances checked
below (8% for The Cannon, 20% for Lux, in $T_{\rm eff}$, $\log g$ and the median abundance).

**Checks.**

In [ ]:
for ours, theirs, tol in [(our_can, px_can, 0.08), (our_lux, px_lux, 0.2)]:
    for j in range(2):
        assert np.all(np.abs(np.array([ours[j][0], ours[j][1], np.median(ours[j][3:])]) /
                             np.array([theirs[j][0], theirs[j][1], np.median(theirs[j][3:])]) - 1) < tol)

## B.5 What pollux adds

Our numpy code does what the chapters need. pollux, in addition, puts priors on every parameter and can sample the
posterior with numpyro instead of optimising; takes any number of outputs, each with its own transform (labels from
two surveys, photometry, spectra from two instruments); allows non-linear transforms from the latents (and then fits
them by gradient descent); and runs on GPUs through JAX. Its documentation has worked examples for APOGEE.

## Summary

- For the least-squares steps of training, pollux and our code agree to $10^{-9}$ of the errors; trained each in its
  own way, the two give the same precision for The Cannon and nearly the same for Lux.
- pollux 0.1.0 needed one pinned dependency (`xmmutablemap` 0.2.2) to run in November 2026; pin versions when you
  build on young packages.

## References

- Horta, D., Price-Whelan, A. M., Hogg, D. W., Ness, M. K., & Casey, A. R. 2025,
  [AJ 169, 314](https://ui.adsabs.harvard.edu/abs/2025AJ....169..314H).
- Ness, M., Hogg, D. W., Rix, H.-W., Ho, A. Y. Q., & Zasowski, G. 2015,
  [ApJ 808, 16](https://ui.adsabs.harvard.edu/abs/2015ApJ...808...16N).
- Price-Whelan, A. M., *pollux*, [github.com/adrn/pollux](https://github.com/adrn/pollux),
  [pollux-astro.readthedocs.io](https://pollux-astro.readthedocs.io).